## 0-a. 패키지 설치 및 Kaggle 데이터셋 다운로드

In [ ]:
%pip install -q kagglehub statsmodels seaborn scikit-learn pandas numpy matplotlib scipy pyarrow

import os
import kagglehub

dataset_path = kagglehub.dataset_download("mkechinov/ecommerce-events-history-in-cosmetics-shop")
print("Dataset downloaded to:", dataset_path)
print(os.listdir(dataset_path))


In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, chi2
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
import glob
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)


## 0-b. CSV 5개월치 로드 & 병합

In [ ]:
csv_files = sorted(glob.glob(os.path.join(dataset_path, "*.csv")))
print(csv_files)

dfs = []
for f in csv_files:
    month_tag = os.path.basename(f).replace(".csv", "")
    tmp = pd.read_csv(f, dtype={"event_type": "category"})
    tmp["event_month"] = month_tag
    dfs.append(tmp)

df_alldata = pd.concat(dfs, ignore_index=True)
df_alldata["event_month"] = df_alldata["event_month"].astype("category")
del dfs
gc.collect()

print(df_alldata.shape)
df_alldata.head()


## 0-c. 기본 정제 + 파생변수 생성

In [ ]:
print(df_alldata.isnull().sum())
print(df_alldata["event_type"].unique())

df = df_alldata.dropna(subset=["user_session"]).copy()
del df_alldata
gc.collect()

df["event_time"] = pd.to_datetime(df["event_time"], utc=True).dt.tz_localize(None)

df["brand"] = df["brand"].astype("category")

cats = df["category_code"].str.split(".", n=2, expand=True)
df["category_l1_raw"] = cats[0].astype("category")
df["category_l2_raw"] = (cats[1] if cats.shape[1] > 1 else pd.Series(np.nan, index=df.index)).astype("category")
df["category_l3_raw"] = (cats[2] if cats.shape[1] > 2 else pd.Series(np.nan, index=df.index)).astype("category")
del cats
gc.collect()

raw_coverage = df["category_l1_raw"].notna().mean()
print(f"\n[확인] category_code 기반 원본 카테고리 커버리지: {raw_coverage:.2%}")
print(f"[확인] category_id 결측 비율: {df['category_id'].isna().mean():.2%}")

df["date"] = df["event_time"].dt.floor("D")
df["hour"] = df["event_time"].dt.hour.astype("int8")
df["dow"] = df["event_time"].dt.dayofweek.astype("int8")
df["weekday_name"] = df["event_time"].dt.day_name().astype("category")
df["is_weekend"] = df["dow"].isin([5, 6])
df["time_bucket"] = pd.cut(
    df["hour"], bins=[-1, 5, 11, 17, 23],
    labels=["dawn", "morning", "afternoon", "evening"]
)  # pd.cut 결과는 이미 Categorical dtype

BLACK_FRIDAY = pd.Timestamp("2019-11-29")
CHRISTMAS = pd.Timestamp("2019-12-25")
VALENTINE = pd.Timestamp("2020-02-14")

df["days_to_bf"] = (df["date"] - BLACK_FRIDAY).dt.days.astype("int16")
df["days_to_xmas"] = (df["date"] - CHRISTMAS).dt.days.astype("int16")
df["days_to_valentine"] = (df["date"] - VALENTINE).dt.days.astype("int16")

df["event_period"] = np.select(
    [
        df["days_to_bf"].between(-7, 1),
        df["days_to_xmas"].between(-7, 1),
        df["days_to_valentine"].between(-7, 1),
    ],
    ["black_friday", "christmas", "valentine"],
    default="normal"
)
df["event_period"] = df["event_period"].astype("category")

gc.collect()
print(df.info(memory_usage="deep"))
df.head()


---
## 0-d. 카테고리 라벨 정리


In [ ]:
df["category_l1"] = df["category_l1_raw"]
df["category_l2"] = df["category_l2_raw"]

coverage = df["category_l1"].notna().mean()
print(f"카테고리 커버리지: {coverage:.2%}")

df = df.drop(columns=["category_l1_raw", "category_l2_raw", "category_l3_raw"])
gc.collect()

df.to_parquet("events_clean.parquet", index=False)
print(df.info(memory_usage="deep"))


---
## 1. 세션 단위 재구성

In [ ]:
sess_pivot = df.pivot_table(index="user_session", columns="event_type",
                             values="user_id", aggfunc="size", fill_value=0, observed=True)

for col in ["view", "cart", "remove_from_cart", "purchase"]:
    if col not in sess_pivot.columns:
        sess_pivot[col] = 0

sess_flags = pd.DataFrame({
    "has_view": sess_pivot["view"] > 0,
    "has_cart": sess_pivot["cart"] > 0,
    "has_remove": sess_pivot["remove_from_cart"] > 0,
    "has_purchase": sess_pivot["purchase"] > 0,
})
del sess_pivot

session_agg = df.groupby("user_session", observed=True).agg(
    user_id=("user_id", "first"),
    n_events=("event_type", "count"),
    n_products=("product_id", "nunique"),
    n_categories=("category_l1", "nunique"),
    session_start=("event_time", "min"),
    session_end=("event_time", "max"),
    avg_price=("price", "mean"),
    n_cart=("event_type", lambda x: (x == "cart").sum()),
).reset_index()

session_agg["dwell_sec"] = (session_agg["session_end"] - session_agg["session_start"]).dt.total_seconds()
session_agg["events_per_product"] = session_agg["n_events"] / session_agg["n_products"].replace(0, np.nan)
session_agg["cart_rate"] = session_agg["n_cart"] / session_agg["n_events"]

for col in ["n_events", "n_products", "n_categories", "n_cart"]:
    session_agg[col] = pd.to_numeric(session_agg[col], downcast="integer")

sessions = session_agg.merge(sess_flags, left_on="user_session", right_index=True, how="left")
del session_agg, sess_flags
gc.collect()

sessions.to_parquet("sessions.parquet", index=False)
sessions.head()


---
## 2. 기본 EDA

In [ ]:
session_event_counts = pd.Series({
    "view_sessions": sessions["has_view"].sum(),
    "cart_sessions": sessions["has_cart"].sum(),
    "remove_sessions": sessions["has_remove"].sum(),
    "purchase_sessions": sessions["has_purchase"].sum(),
})
session_event_counts.plot.bar(figsize=(8,5), title="Session-based Event Counts")
plt.show(); plt.close("all")

p99 = df["price"].quantile(0.99)
plt.figure(figsize=(8,5))
df.loc[df["price"] <= p99, "price"].hist(bins=80)
plt.title(f"Price Distribution (clipped at 99th pct = {p99:.1f})")
plt.show(); plt.close("all")

df["category_l1"].value_counts().head(10).plot.barh(figsize=(8,5), title="Top 10 category_l1")
plt.gca().invert_yaxis()
plt.show(); plt.close("all")

df["brand"].value_counts().head(10).plot.barh(figsize=(8,5), title="Top 10 Brand")
plt.gca().invert_yaxis()
plt.show(); plt.close("all")


In [ ]:
heat_activity = df.pivot_table(index="dow", columns="hour", values="event_type", aggfunc="count")
plt.figure(figsize=(14,5))
sns.heatmap(heat_activity, cmap="YlOrRd")
plt.title("Activity Heatmap (dow x hour)")
plt.show(); plt.close("all")
del heat_activity

purchase_only = df.loc[df["event_type"] == "purchase", ["dow", "hour"]]
view_only_ev = df.loc[df["event_type"] == "view", ["dow", "hour"]]
conv_by_hour_dow = (
    purchase_only.groupby(["dow","hour"]).size() / view_only_ev.groupby(["dow","hour"]).size()
).unstack()
del purchase_only, view_only_ev
plt.figure(figsize=(14,5))
sns.heatmap(conv_by_hour_dow, cmap="YlGnBu")
plt.title("Conversion Rate Heatmap (dow x hour, purchase/view ratio) - [주의] 이벤트 개수 기준, 왜곡 가능성 있음, 아래 2-보강에서 세션 기준으로 재검증")
plt.show(); plt.close("all")

funnel_basic = pd.DataFrame({
    "stage": ["View","Cart","Purchase"],
    "sessions": [sessions["has_view"].sum(), sessions["has_cart"].sum(), sessions["has_purchase"].sum()]
})
funnel_basic["conversion_from_view_%"] = funnel_basic["sessions"] / funnel_basic["sessions"].iloc[0] * 100
gc.collect()
funnel_basic


---
## 2-보강. 세션 기준 시간대 전환율 (이벤트 개수 왜곡 없이 검증)

In [ ]:
sessions["session_hour"] = sessions["session_start"].dt.hour
sessions["session_dow"] = sessions["session_start"].dt.dayofweek

session_activity = sessions.pivot_table(
    index="session_dow", columns="session_hour", values="user_session", aggfunc="count"
)
plt.figure(figsize=(14,5))
sns.heatmap(session_activity, cmap="YlOrRd")
plt.title("Session Count Heatmap (dow x hour) - 세션 기준 표본 크기")
plt.show(); plt.close("all")

session_conv = (
    sessions[sessions["has_view"]]
    .groupby(["session_dow","session_hour"])["has_purchase"]
    .mean()
    .unstack()
)
session_conv_n = (
    sessions[sessions["has_view"]]
    .groupby(["session_dow","session_hour"])["has_purchase"]
    .count()
    .unstack()
)
MIN_N = 30
mask = session_conv_n < MIN_N
print(f"표본이 {MIN_N}건 미만인 셀: {(session_conv_n < MIN_N).sum().sum()} / 전체 {session_conv_n.size}칸")

plt.figure(figsize=(14,5))
sns.heatmap(session_conv, cmap="YlGnBu", mask=mask)
plt.title(f"Session-based Conversion Rate (n>={MIN_N}인 셀만 표시)")
plt.show(); plt.close("all")
del session_activity, session_conv_n
gc.collect()


---
## 3. 퍼널 심화: Nested vs Non-nested -> Immediate / Delayed / Direct
참고: https://www.kaggle.com/code/phantirasea/e-commerce-funnel-customer-decision-analysis

In [ ]:
V = sessions["has_view"].sum()
VC = (sessions["has_view"] & sessions["has_cart"]).sum()
VCP = (sessions["has_view"] & sessions["has_cart"] & sessions["has_purchase"]).sum()

nested_funnel = pd.DataFrame({"stage": ["V","VC","VCP"], "sessions": [V, VC, VCP]})
nested_funnel["conversion_to_next_%"] = nested_funnel["sessions"].shift(-1) / nested_funnel["sessions"] * 100
print(nested_funnel)

non_nested = pd.Series({
    "View": sessions["has_view"].sum(),
    "Cart": sessions["has_cart"].sum(),
    "Purchase": sessions["has_purchase"].sum(),
})
funnel_compare = pd.DataFrame({"non_nested": non_nested.values, "nested": [V, VC, VCP]},
                               index=["View","Cart","Purchase"])
funnel_compare["gap"] = funnel_compare["non_nested"] - funnel_compare["nested"]
funnel_compare


In [ ]:
purchase_sessions = (
    df.loc[df["event_type"]=="purchase", ["user_session","user_id","event_time"]]
    .groupby(["user_session","user_id"], as_index=False)
    .agg(purchase_time=("event_time","min"))
)
purchase_sessions = purchase_sessions.merge(sessions[["user_session","has_view","has_cart","has_purchase"]],
                                             on="user_session", how="left")

purchase_sessions["group"] = np.where(
    purchase_sessions["has_view"] & purchase_sessions["has_cart"] & purchase_sessions["has_purchase"],
    "Immediate", "Other"
)

viewcart = df.loc[df["event_type"].isin(["view","cart"]), ["user_id","event_time"]]
first_viewcart_time = viewcart.groupby("user_id")["event_time"].min().rename("first_viewcart_time")
del viewcart
purchase_sessions = purchase_sessions.merge(first_viewcart_time, on="user_id", how="left")
del first_viewcart_time

mask_other = purchase_sessions["group"].eq("Other")
mask_delayed = mask_other & purchase_sessions["first_viewcart_time"].notna() & \
               (purchase_sessions["first_viewcart_time"] < purchase_sessions["purchase_time"])
purchase_sessions.loc[mask_delayed, "group"] = "Delayed"
purchase_sessions.loc[mask_other & ~mask_delayed, "group"] = "Direct"

purchase_revenue = df.loc[df["event_type"]=="purchase", ["user_session","price"]].groupby("user_session")["price"].sum().rename("revenue")
purchase_sessions = purchase_sessions.merge(purchase_revenue, on="user_session", how="left")
del purchase_revenue
gc.collect()

print(purchase_sessions["group"].value_counts(normalize=True) * 100)
purchase_sessions.head()


In [ ]:
purchase_session_metrics = (
    df.loc[df["event_type"]=="purchase", ["user_session","price","product_id"]]
    .groupby(["user_session"], as_index=False)
    .agg(revenue_session=("price","sum"), items_session=("product_id","count"))
    .merge(purchase_sessions[["user_session","group"]], on="user_session", how="left")
)

summary_by_group = purchase_session_metrics.groupby("group").agg(
    sessions=("user_session","nunique"),
    revenue=("revenue_session","sum"),
    AOV=("revenue_session","mean"),
    avg_basket_size=("items_session","mean"),
).reset_index()
print(summary_by_group)
del purchase_session_metrics

p_purchase = df.loc[df["event_type"]=="purchase", ["user_session","price","event_month"]].merge(
    purchase_sessions[["user_session","group"]], on="user_session", how="left")
monthly_rev = p_purchase.groupby(["event_month","group"], observed=True)["price"].sum().unstack().fillna(0)
del p_purchase
monthly_rev.plot(figsize=(12,5), marker="o", title="Monthly Revenue by Purchase-Path Group")
plt.show(); plt.close("all")
gc.collect()


---
## 4. 세션 아키타입 클러스터링 (K-means)

In [ ]:
sessions_multi = sessions[sessions["n_events"] > 1].copy()

cluster_features = ["n_events","n_products","n_categories","dwell_sec",
                     "events_per_product","cart_rate","avg_price"]

log_cols = ["n_events", "n_products", "dwell_sec", "avg_price"]
X = sessions_multi[cluster_features].fillna(0).copy()
for col in log_cols:
    X[col] = np.log1p(X[col])
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

Xs = StandardScaler().fit_transform(X)
del X

ks = range(2, 9)
inertia, sil = [], []
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(Xs), size=min(10000, len(Xs)), replace=False)

for k in ks:
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=10, batch_size=10000)
    labels = km.fit_predict(Xs)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xs[sample_idx], labels[sample_idx]))

fig, ax = plt.subplots(1,2, figsize=(11,4))
ax[0].plot(list(ks), inertia, "o-"); ax[0].set_title("Elbow (inertia, log-transformed)")
ax[1].plot(list(ks), sil, "o-", color="darkorange"); ax[1].set_title("Silhouette score (log-transformed)")
plt.show(); plt.close("all")


In [ ]:
K_FINAL = 4  # 엘보우/실루엣 둘 다 지지하는 값 (필요 시 위 그래프 보고 조정)

km_final = MiniBatchKMeans(n_clusters=K_FINAL, random_state=42, n_init=20, batch_size=10000)
sessions_multi["archetype"] = km_final.fit_predict(Xs)

archetype_profile = sessions_multi.groupby("archetype")[cluster_features].mean()
archetype_profile["n_sessions"] = sessions_multi["archetype"].value_counts()
archetype_profile["conversion_rate_%"] = sessions_multi.groupby("archetype")["has_purchase"].mean() * 100

n_realized = sessions_multi["archetype"].nunique()
print(f"요청한 K={K_FINAL}, 실제로 만들어진 군집 수={n_realized}")
if n_realized < K_FINAL:
    print("[경고] 여전히 일부 군집이 비어있습니다.")

del Xs, sample_idx
gc.collect()

archetype_profile


---
## 5. 특이일(이벤트데이) 분석

In [ ]:
session_period = df.groupby("user_session", observed=True)["event_period"].agg(lambda x: x.mode().iloc[0])
sessions_multi = sessions_multi.merge(session_period.rename("event_period"),
                                       left_on="user_session", right_index=True, how="left")

event_period_summary = sessions_multi.groupby("event_period", observed=True).agg(
    sessions=("user_session","nunique"),
    conversion_rate=("has_purchase","mean"),
    avg_price=("avg_price","mean"),
).reset_index()
print(event_period_summary)

archetype_by_period = pd.crosstab(sessions_multi["event_period"], sessions_multi["archetype"], normalize="index") * 100
archetype_by_period.plot(kind="bar", stacked=True, figsize=(10,5), title="Archetype Mix by Event Period")
plt.show(); plt.close("all")


In [ ]:
purchase_sessions_period = purchase_sessions.merge(
    session_period.rename("event_period"), left_on="user_session", right_index=True, how="left")

group_by_period = pd.crosstab(purchase_sessions_period["event_period"], purchase_sessions_period["group"],
                               normalize="index") * 100
group_by_period.plot(kind="bar", stacked=True, figsize=(10,5),
                      title="Immediate/Delayed/Direct Mix by Event Period")
plt.show(); plt.close("all")
del purchase_sessions_period
gc.collect()


---
## 7. 카테고리별 구매여부 판별 피처 검정 (Mann-Whitney U)

In [ ]:
sessions_labeled = sessions.copy()
main_category = df.groupby("user_session", observed=True)["category_l1"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
sessions_labeled = sessions_labeled.merge(main_category.rename("main_category"),
                                           left_on="user_session", right_index=True, how="left")

cat_labeled_share = sessions_labeled["main_category"].notna().mean()
print(f"[참고] 카테고리 정보가 있는 세션 비율: {cat_labeled_share:.2%}")

candidate_features = ["avg_price","dwell_sec","n_products","n_categories",
                       "events_per_product","cart_rate"]

results = []
for cat, g in sessions_labeled.dropna(subset=["main_category"]).groupby("main_category", observed=True):
    purchased = g[g["has_purchase"]]
    not_purchased = g[~g["has_purchase"]]
    for feat in candidate_features:
        x = purchased[feat].dropna()
        y = not_purchased[feat].dropna()
        if len(x) < 10 or len(y) < 10:
            continue
        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        effect_size = (2*stat) / (len(x)*len(y)) - 1
        direction = "구매그룹이 더 큼" if x.median() > y.median() else (
            "구매그룹이 더 작음" if x.median() < y.median() else "동일")
        results.append({
            "category": cat, "feature": feat, "U": stat, "p_value": pval,
            "effect_size": effect_size, "direction": direction,
            "median_purchase": x.median(), "median_non_purchase": y.median(),
            "n_purchase": len(x), "n_non_purchase": len(y),
        })

result_df = pd.DataFrame(results)
result_df["p_adj"] = multipletests(result_df["p_value"], method="fdr_bh")[1]
result_df = result_df.reindex(result_df["effect_size"].abs().sort_values(ascending=False).index)
result_df.to_csv("mannwhitney_feature_results.csv", index=False)
gc.collect()
result_df.head(20)


---
## 8. 카테고리 × 피처 상호작용 검정

In [ ]:
interaction_base = sessions_labeled.dropna(subset=["main_category"]).copy()
interaction_base["y"] = interaction_base["has_purchase"].astype(int)

interaction_results = []
for feat in candidate_features:
    sub = interaction_base[[feat, "main_category", "y"]].dropna().copy()
    if sub[feat].std() == 0 or len(sub) < 200:
        print(f"{feat}: 표본 부족 또는 분산 0, 스킵")
        continue
    sub["z"] = (sub[feat] - sub[feat].mean()) / sub[feat].std()

    try:
        m_full = smf.logit("y ~ z * C(main_category)", data=sub).fit(disp=0, maxiter=100)
        m_reduced = smf.logit("y ~ z + C(main_category)", data=sub).fit(disp=0, maxiter=100)
        lr_stat = 2 * (m_full.llf - m_reduced.llf)
        df_diff = m_full.df_model - m_reduced.df_model
        lr_pvalue = chi2.sf(lr_stat, df_diff)
        del m_full, m_reduced
    except Exception as e:
        lr_stat, df_diff, lr_pvalue = np.nan, np.nan, np.nan
        print(f"{feat}: 회귀 실패 ({e})")
    del sub

    interaction_results.append({
        "feature": feat, "lr_stat": lr_stat, "df_diff": df_diff, "lr_pvalue": lr_pvalue
    })

gc.collect()
interaction_df = pd.DataFrame(interaction_results)
interaction_df["p_adj"] = multipletests(interaction_df["lr_pvalue"].fillna(1), method="fdr_bh")[1]
interaction_df


In [ ]:
heterogeneity = (
    result_df.groupby("feature")["effect_size"]
    .agg(mean_effect="mean", std_effect="std", n_categories="count")
    .reset_index()
    .sort_values("std_effect", ascending=False)
)
heterogeneity


In [ ]:
INTERACTION_ALPHA = 0.05
significant_interaction_features = interaction_df.loc[
    interaction_df["p_adj"] < INTERACTION_ALPHA, "feature"].tolist()
pooled_features = [f for f in candidate_features if f not in significant_interaction_features]

print("카테고리별 상호작용 유의:", significant_interaction_features)
print("상호작용 비유의:", pooled_features)


---
## 9-a. 카테고리별 심화 '왜 안 사는지'


In [ ]:
needed_cols = ["user_session", "product_id", "event_type", "price", "category_l1", "hour"]
prod_events = df[needed_cols]

sp_pivot = prod_events.pivot_table(
    index=["user_session","product_id"], columns="event_type",
    values="price", aggfunc="size", fill_value=0, observed=True
)
for col in ["view","cart","remove_from_cart","purchase"]:
    if col not in sp_pivot.columns:
        sp_pivot[col] = 0

sp_flags = pd.DataFrame({
    "has_purchase": sp_pivot["purchase"] > 0,
    "has_remove": sp_pivot["remove_from_cart"] > 0,
    "has_cart": sp_pivot["cart"] > 0,
}, index=sp_pivot.index).reset_index()
del sp_pivot

conditions = [sp_flags["has_purchase"], sp_flags["has_remove"], sp_flags["has_cart"]]
choices = ["purchased", "removed", "cart_no_purchase"]
sp_flags["intent"] = np.select(conditions, choices, default="view_only")

session_product_status = sp_flags[["user_session","product_id","intent"]]
del sp_flags
gc.collect()

product_features = prod_events.groupby(["user_session","product_id"]).agg(
    price=("price","first"),
    category_l1=("category_l1","first"),
    hour=("hour","first"),
    n_views=("event_type", lambda x: (x=="view").sum()),
).reset_index()
del prod_events

session_product_status = session_product_status.merge(
    product_features, on=["user_session","product_id"], how="left")
del product_features

session_product_status["price_pct_in_category"] = (
    session_product_status.groupby("category_l1", observed=True)["price"].rank(pct=True))

concurrent = session_product_status.groupby("user_session")["product_id"].nunique().rename(
    "concurrent_products_in_session")
session_product_status = session_product_status.merge(
    concurrent, on="user_session", how="left")
del concurrent
gc.collect()

session_product_status["intent"].value_counts()


In [ ]:
session_to_product_feature_map = {
    "avg_price": "price_pct_in_category",
    "n_products": "concurrent_products_in_session",
    "events_per_product": "n_views",
    "dwell_sec": None,
    "cart_rate": None,
    "n_categories": None,
}

mapped_significant = {session_to_product_feature_map[f]
                       for f in significant_interaction_features
                       if session_to_product_feature_map.get(f) is not None}
mapped_significant |= {"price_pct_in_category"}
mapped_significant = sorted(mapped_significant)

print("카테고리별 심화 '왜 안 사는지' 검정에 쓸 상품단위 피처:", mapped_significant)

target_pairs = [("purchased","cart_no_purchase"), ("cart_no_purchase","removed")]

why_results_by_category = []
for cat, g in session_product_status.dropna(subset=["category_l1"]).groupby("category_l1", observed=True):
    for grp_a, grp_b in target_pairs:
        a = g[g["intent"]==grp_a]
        b = g[g["intent"]==grp_b]
        for feat in mapped_significant:
            x = a[feat].dropna()
            y = b[feat].dropna()
            if len(x) < 10 or len(y) < 10:
                continue
            stat, pval = mannwhitneyu(x, y, alternative="two-sided")
            effect_size = (2*stat) / (len(x)*len(y)) - 1
            direction = "앞 그룹이 더 큼" if x.median() > y.median() else (
                "앞 그룹이 더 작음" if x.median() < y.median() else "동일")
            why_results_by_category.append({
                "category": cat, "compare": f"{grp_a}_vs_{grp_b}", "feature": feat,
                "p_value": pval, "effect_size": effect_size, "direction": direction,
                "median_a": x.median(), "median_b": y.median(),
            })

why_df_by_category = pd.DataFrame(why_results_by_category)
if len(why_df_by_category):
    why_df_by_category["p_adj"] = multipletests(why_df_by_category["p_value"], method="fdr_bh")[1]
    why_df_by_category = why_df_by_category.reindex(
        why_df_by_category["effect_size"].abs().sort_values(ascending=False).index)
why_df_by_category.to_csv("why_not_buy_by_category.csv", index=False)
gc.collect()
why_df_by_category.head(20)


---
## 9-b. 전체 데이터 풀링 '왜 안 사는지'

In [ ]:
pooled_mapped = {session_to_product_feature_map[f]
                  for f in pooled_features
                  if session_to_product_feature_map.get(f) is not None}
pooled_mapped |= {"n_views", "hour"}
pooled_mapped = sorted(pooled_mapped)

print("전체 풀링 '왜 안 사는지' 검정에 쓸 피처:", pooled_mapped)

why_results_pooled = []
g_all = session_product_status
for grp_a, grp_b in target_pairs:
    a = g_all[g_all["intent"]==grp_a]
    b = g_all[g_all["intent"]==grp_b]
    for feat in pooled_mapped:
        x = a[feat].dropna()
        y = b[feat].dropna()
        if len(x) < 10 or len(y) < 10:
            continue
        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        effect_size = (2*stat) / (len(x)*len(y)) - 1
        direction = "앞 그룹이 더 큼" if x.median() > y.median() else (
            "앞 그룹이 더 작음" if x.median() < y.median() else "동일")
        why_results_pooled.append({
            "compare": f"{grp_a}_vs_{grp_b}", "feature": feat,
            "p_value": pval, "effect_size": effect_size, "direction": direction,
            "median_a": x.median(), "median_b": y.median(),
        })

why_df_pooled = pd.DataFrame(why_results_pooled)
if len(why_df_pooled):
    why_df_pooled["p_adj"] = multipletests(why_df_pooled["p_value"], method="fdr_bh")[1]
    why_df_pooled = why_df_pooled.reindex(
        why_df_pooled["effect_size"].abs().sort_values(ascending=False).index)
why_df_pooled.to_csv("why_not_buy_pooled.csv", index=False)
why_df_pooled
